# MakharijPro AI — Track A / Phase 5: QDAT Dataset Manifest

**Run this on Kaggle** (GPU not required for this notebook, but leave Internet ON to pull QDAT).

## What Phase 1-4 established (from `audit_report.json`, two audit rounds)

- QDAT: 1,505 clips, 16kHz mono, 0 corrupted, **108 exact-audio duplicates**.
- Direct inspection of duplicate pairs (not guessed) showed three patterns:
  1. Most pairs: only `id`/`original_id` differ — true redundant rows, safe to drop one copy.
  2. A few pairs: one rule-label field flips while audio + everything else is identical
     (e.g. `separate_tide` 0 vs 1) — annotation noise on identical audio.
  3. A few pairs: **different speakers** (different `original_id` prefix, different `age`) but
     **bit-identical audio** — most likely a data-construction defect in QDAT itself (same audio
     linked to two speaker entries), not a real coincidence.
- Label semantics (best-supported hypothesis, not 100% certain — HF/ResearchGate fetches were
  blocked by network resets when trying to confirm against the original paper):
  `separate_tide` = Separate Madd correct(1)/incorrect(0), `the_tight_noon` = Ghunnah-triggering
  "tight noon" correct/incorrect, `concealment` = Ikhfa correct/incorrect. Published QDAT results
  report **three independent per-rule accuracies** (96%/95%/96%), which matches treating these three
  columns as three separate binary targets rather than one combined `target`. `target`'s exact
  definition stays unresolved — this notebook keeps it in the manifest but does not treat it as a
  primary label, so we're not training against a column we don't understand.
- `original_id` (e.g. `s100_8`) encodes a **speaker id** (`s100`) — this is the grouping key that
  MUST be used for train/val/test splitting to avoid the leakage your own spec (§6) warns about.
  Random per-clip splitting would put the same speaker's voice in both train and test.

## What this notebook produces

`qdat_manifest.csv` / `.parquet` — one row per **unique** audio clip (duplicates resolved), with a
`speaker_id` column and a leakage-safe, speaker-grouped train/val/test split. This is the artifact
Phase 6 (feature extraction) consumes next — save it as a Kaggle Dataset output so it survives the
session.

## 1. Load QDAT and compute a stable content hash per row

In [ ]:
from datasets import load_dataset
import numpy as np
import hashlib
import pandas as pd
import re

qdat = load_dataset("obadx/qdat")["train"]
print(f"Loaded {len(qdat)} rows")

rows = []
for i, ex in enumerate(qdat):
    audio = ex["audio"]
    arr = np.asarray(audio["array"], dtype=np.float32)
    content_hash = hashlib.md5(arr.tobytes()).hexdigest()
    rows.append({
        "row_index": i,
        "content_hash": content_hash,
        "duration_seconds": len(arr) / audio["sampling_rate"],
        "sample_rate": audio["sampling_rate"],
        "id": ex["id"],
        "original_id": ex["original_id"],
        "age": ex["age"],
        "gender": ex["gender"],
        "target": ex["target"],
        "separate_tide": ex["separate_tide"],
        "the_tight_noon": ex["the_tight_noon"],
        "concealment": ex["concealment"],
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

## 2. Extract speaker_id from `original_id` — verify the pattern actually holds

Do not assume every row matches `s<digits>_<digits>`; count and report any that don't, per the
"don't fabricate, report conflicts" rule.

In [ ]:
pattern = re.compile(r"^s(\d+)_(\d+)$")
matches = df["original_id"].apply(lambda x: pattern.match(str(x)))
n_unmatched = matches.isna().sum()
print(f"original_id rows matching 's<speaker>_<utterance>' pattern: {len(df) - n_unmatched}/{len(df)}")
if n_unmatched > 0:
    print("Unmatched examples:", df.loc[matches.isna(), "original_id"].head(20).tolist())

df["speaker_id"] = matches.apply(lambda m: m.group(1) if m else None)
df["utterance_id"] = matches.apply(lambda m: m.group(2) if m else None)

print(f"\nUnique speakers found: {df['speaker_id'].nunique()}")
print(df["speaker_id"].value_counts().describe())

## 3. Resolve duplicate-audio groups

For each group of rows sharing the same `content_hash`:
- keep exactly one canonical row (lowest `row_index`),
- for each label column, if all rows in the group agree, keep that value,
- if they disagree, resolve by majority vote and flag the field as `<field>_had_conflict=True`
  so this is visible later rather than silently hidden,
- if the group spans more than one `speaker_id`, flag `cross_speaker_duplicate=True` — this is the
  data-defect pattern found in Phase 4 and should not be trusted for speaker-level statistics.

In [ ]:
LABEL_COLS = ["target", "separate_tide", "the_tight_noon", "concealment"]

def resolve_group(g):
    canonical = g.sort_values("row_index").iloc[0].to_dict()
    canonical["n_duplicates_in_group"] = len(g)
    canonical["cross_speaker_duplicate"] = g["speaker_id"].nunique(dropna=False) > 1
    for col in LABEL_COLS:
        vals = g[col].dropna()
        if vals.nunique() <= 1:
            canonical[f"{col}_had_conflict"] = False
        else:
            canonical[col] = vals.mode().iloc[0]  # majority vote; ties -> first mode
            canonical[f"{col}_had_conflict"] = True
    return canonical

# Plain groupby iteration, not .groupby().apply() -- sidesteps the pandas deprecation around
# grouping columns being dropped from what's passed into an applied function, since we
# deliberately want content_hash available inside resolve_group.
resolved = pd.DataFrame([resolve_group(g) for _, g in df.groupby("content_hash")]).reset_index(drop=True)

n_dropped = len(df) - len(resolved)
print(f"Rows before dedup: {len(df)}  |  after dedup: {len(resolved)}  |  dropped: {n_dropped}")

conflict_cols = [c for c in resolved.columns if c.endswith("_had_conflict")]
for c in conflict_cols:
    n = int(resolved[c].sum())
    if n:
        print(f"  {c}: {n} groups had disagreeing labels (resolved by majority vote)")

n_cross_speaker = int(resolved["cross_speaker_duplicate"].sum())
print(f"\nCross-speaker duplicate groups (same audio, different speaker_id): {n_cross_speaker}")
if n_cross_speaker:
    print(resolved.loc[resolved["cross_speaker_duplicate"],
                        ["row_index", "speaker_id", "original_id", "n_duplicates_in_group"]])

## 4. Class balance check (per label, on the deduplicated set)

Needed before Phase 9/10 — an 80/20 imbalance changes which metric (F1 vs. raw accuracy) is
meaningful, per your own Phase 10 requirement.

In [ ]:
for col in LABEL_COLS:
    print(f"\n{col}:")
    print(resolved[col].value_counts(dropna=False))
    print(f"  imbalance ratio (majority/minority): "
          f"{resolved[col].value_counts().max() / resolved[col].value_counts().min():.2f}")

## 5. Speaker-grouped train/val/test split

Splitting by `speaker_id`, not by row, so no speaker's voice appears in more than one split —
this is the leakage-prevention requirement from your own spec, now applied for real. Cross-speaker
duplicate rows (flagged above) are dropped entirely rather than assigned to a speaker group we
can't trust.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42
clean = resolved.loc[~resolved["cross_speaker_duplicate"]].copy()
n_excluded = len(resolved) - len(clean)
print(f"Excluded {n_excluded} cross-speaker-duplicate rows from the splittable set "
      f"({len(clean)} rows remain).")

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
train_idx, temp_idx = next(gss1.split(clean, groups=clean["speaker_id"]))
train_df = clean.iloc[train_idx].copy()
temp_df = clean.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["speaker_id"]))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part["split"] = name

manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Verify: no speaker appears in more than one split.
speaker_split_counts = manifest.groupby("speaker_id")["split"].nunique()
leaking_speakers = speaker_split_counts[speaker_split_counts > 1]
print(f"Speakers appearing in >1 split (must be 0): {len(leaking_speakers)}")
assert len(leaking_speakers) == 0, "Leakage detected — a speaker appears in multiple splits."

print(manifest["split"].value_counts())
for col in LABEL_COLS:
    print(f"\n{col} distribution by split:")
    print(manifest.groupby("split")[col].value_counts(normalize=True).round(3))

## 6. Save the manifest

Save as both CSV (human-inspectable) and Parquet (preserves dtypes for Phase 6). On Kaggle, this
needs to be published as a Kaggle Dataset output (or attached via "Save Version") so the next
notebook can read it without recomputing everything.

In [ ]:
manifest_cols = [
    "row_index", "id", "original_id", "speaker_id", "utterance_id", "age", "gender",
    "duration_seconds", "sample_rate", "target", "separate_tide", "the_tight_noon", "concealment",
    "n_duplicates_in_group", "split",
] + conflict_cols

manifest[manifest_cols].to_csv("qdat_manifest.csv", index=False)
manifest[manifest_cols].to_parquet("qdat_manifest.parquet", index=False)

print("Saved qdat_manifest.csv and qdat_manifest.parquet")
print(f"\nFinal manifest: {len(manifest)} unique clips "
      f"({len(df) - len(manifest)} removed: {n_dropped} exact duplicates + {n_excluded} cross-speaker conflicts)")
manifest[manifest_cols].head(10)

## 7. Phase 6 — Canonical feature extraction (MFCC + delta)

Real run results confirmed the manifest is sound: 108 dropped matches Phase 1's duplicate count
exactly, 0 speaker leakage across splits. Moving straight to features.

Baseline per SDD §6.1, not deviated from: 13 MFCC coefficients + first-order delta (26 dims total),
25ms frame / 10ms hop, computed at the native 16kHz. Two decisions the SDD didn't pin down, made
explicit here rather than left implicit:
- **Per-clip peak amplitude normalization + silence trim (top_db=25)** before MFCC, so loudness
  differences between recording setups don't masquerade as pronunciation signal.
- **No fixed-length padding here.** Clips run 2.3–17.5s; padding every clip to the longest one now
  would waste memory and bake in a batching decision that belongs to Phase 9 (`tf.data`'s
  `padded_batch` handles variable length per-batch far more efficiently). Each clip's feature
  matrix is saved at its natural length; shape is recorded in the index for exactly this reason.
- **Feature normalization stats (mean/std) are fit on the TRAIN split only**, then saved — applying
  train-only statistics to val/test is the same leakage discipline as the speaker-grouped split,
  now extended to feature scaling.

Every clip is written to disk as it's processed (not accumulated in a Python list) and skipped if
its `.npy` already exists — resumable if the session drops, per your own spec.

In [ ]:
import librosa
import numpy as np
import os

FEATURE_DIR = "features"
os.makedirs(FEATURE_DIR, exist_ok=True)

FEATURE_CONFIG = {
    "sample_rate": 16000,
    "n_mfcc": 13,
    "frame_length_ms": 25,
    "hop_length_ms": 10,
    "frame_length_samples": int(16000 * 0.025),
    "hop_length_samples": int(16000 * 0.010),
    "delta_order": 1,
    "trim_top_db": 25,
    "feature_dim": 13 * 2,
    "librosa_version": librosa.__version__,
}
print(FEATURE_CONFIG)

def extract_mfcc_features(y, sr, config=FEATURE_CONFIG):
    assert sr == config["sample_rate"], f"Expected {config['sample_rate']}Hz, got {sr}"
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    y_trimmed, _ = librosa.effects.trim(y, top_db=config["trim_top_db"])
    if len(y_trimmed) < config["frame_length_samples"]:
        y_trimmed = y  # near-silent clip after trim -- fall back to untrimmed rather than error
    mfcc = librosa.feature.mfcc(
        y=y_trimmed, sr=sr, n_mfcc=config["n_mfcc"],
        n_fft=config["frame_length_samples"], hop_length=config["hop_length_samples"],
    )
    delta = librosa.feature.delta(mfcc, order=config["delta_order"])
    features = np.concatenate([mfcc, delta], axis=0)  # (26, T)
    return features.T.astype(np.float32)  # (T, 26) -- time-major, ready for padded_batch later

In [ ]:
def process_manifest(manifest_df, qdat_dataset, feature_dir=FEATURE_DIR, config=FEATURE_CONFIG):
    rows = []
    for n, (_, row) in enumerate(manifest_df.iterrows(), start=1):
        out_path = os.path.join(feature_dir, f"{row['id']}.npy")
        if os.path.exists(out_path):
            feat = np.load(out_path)
        else:
            ex = qdat_dataset[int(row["row_index"])]
            y = np.asarray(ex["audio"]["array"], dtype=np.float32)
            sr = ex["audio"]["sampling_rate"]
            feat = extract_mfcc_features(y, sr, config)
            np.save(out_path, feat)
        rows.append({"id": row["id"], "npy_path": out_path, "n_frames": feat.shape[0], "feature_dim": feat.shape[1]})
        if n % 200 == 0 or n == len(manifest_df):
            print(f"  processed {n}/{len(manifest_df)}")
    return pd.DataFrame(rows)

feature_index = process_manifest(manifest, qdat)
feature_manifest = manifest.merge(feature_index, on="id", how="left")
print(f"\n{feature_manifest['npy_path'].notna().sum()}/{len(feature_manifest)} clips have features")
print(feature_manifest[["n_frames", "feature_dim"]].describe())

### 7b. Normalization stats — fit on TRAIN split only

In [ ]:
train_ids = feature_manifest.loc[feature_manifest["split"] == "train", "id"].tolist()
train_feats = [np.load(os.path.join(FEATURE_DIR, f"{i}.npy")) for i in train_ids]
concat = np.concatenate(train_feats, axis=0)
feat_mean = concat.mean(axis=0)
feat_std = concat.std(axis=0) + 1e-8

FEATURE_CONFIG["normalization_mean"] = feat_mean.tolist()
FEATURE_CONFIG["normalization_std"] = feat_std.tolist()
FEATURE_CONFIG["normalization_fit_on"] = f"train split only, n={len(train_ids)} clips"

print("Per-dimension mean (first 5):", feat_mean[:5])
print("Per-dimension std (first 5):", feat_std[:5])
del concat, train_feats  # free the ~concatenated array once stats are extracted

### 7c. Save the feature manifest and config — the Phase 6 deliverable

In [ ]:
import json as json_module  # local alias avoids shadowing the `json` used earlier in this notebook

feature_manifest.to_csv("feature_manifest.csv", index=False)
feature_manifest.to_parquet("feature_manifest.parquet", index=False)
with open("feature_config.json", "w") as f:
    json_module.dump(FEATURE_CONFIG, f, indent=2)

print("Saved feature_manifest.csv/.parquet and feature_config.json")
print(f"Feature files: {len(os.listdir(FEATURE_DIR))} .npy files in {FEATURE_DIR}/")
feature_manifest[["id", "split", "n_frames", "feature_dim", "target", "separate_tide",
                   "the_tight_noon", "concealment"]].head(10)

### 7d. Save one summary file to send back

Everything Phase 9 needs to know about this run, in one small JSON — download this file
(`phase6_summary.json`) from Kaggle's Output panel and send it back, same as `audit_report.json`
before. No need to paste raw cell output.

In [ ]:
phase6_summary = {
    "n_clips_in_manifest": len(feature_manifest),
    "n_clips_with_features": int(feature_manifest["npy_path"].notna().sum()),
    "n_npy_files_on_disk": len(os.listdir(FEATURE_DIR)),
    "frame_count_stats": feature_manifest["n_frames"].describe().to_dict(),
    "feature_dim": int(feature_manifest["feature_dim"].dropna().iloc[0]) if feature_manifest["feature_dim"].notna().any() else None,
    "split_counts": feature_manifest["split"].value_counts().to_dict(),
    "label_nan_counts": {col: int(feature_manifest[col].isna().sum()) for col in LABEL_COLS},
    "feature_config": FEATURE_CONFIG,
}

with open("phase6_summary.json", "w") as f:
    json_module.dump(phase6_summary, f, indent=2, default=str)

print("Saved phase6_summary.json -- download this one file and send it back.")
print(json_module.dumps({k: v for k, v in phase6_summary.items() if k != "feature_config"}, indent=2, default=str))

## 8. Phase 9 — Baseline multi-task model

QDAT has no word-level boundaries or labels anywhere in its schema — only per-clip labels. The
SDD's `detectTajweedErrors` PDL assumes per-word alignment against a reference vector, which this
data cannot support. So the baseline here is what the data actually allows: a **clip-level**
classifier (whole-recitation features → correct/incorrect per rule), not a word-level comparator.
That's a data constraint, not a preference — noted here for the record, not re-litigated later.

Three usable tasks (`target` stays excluded — meaning still unconfirmed): `separate_tide`,
`the_tight_noon`, `concealment`. One shared small CNN backbone with three sigmoid heads, rather
than three separate models — with only 922 training clips, three independent models would each
starve; sharing representation across tasks is the standard small-data move.

`the_tight_noon` has 1 `NaN` label — handled via a per-row sample weight of 0 for that task only
(the clip still contributes to the other two tasks), not dropped from the dataset. Class imbalance
(`the_tight_noon` 3.95:1) is handled the same way — inverse-frequency sample weights computed from
the TRAIN split only.

In [ ]:
import tensorflow as tf

RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)  # also seeds Python's random and NumPy -- without
# this, weight init and dropout masks differ run-to-run even with the data split/shuffle seeded,
# which is exactly why two runs of this notebook gave different numbers. Fixed going forward.

TASKS = ["separate_tide", "the_tight_noon", "concealment"]
norm_mean = np.array(FEATURE_CONFIG["normalization_mean"], dtype=np.float32)
norm_std = np.array(FEATURE_CONFIG["normalization_std"], dtype=np.float32)

# Inverse-frequency class weights, fit on TRAIN split only -- same leakage discipline as everything else.
class_weights = {}
train_sub = feature_manifest[feature_manifest["split"] == "train"]
for col in TASKS:
    vc = train_sub[col].value_counts()
    total = vc.sum()
    class_weights[col] = {int(k): float(total / (2 * v)) for k, v in vc.items()}
print("Class weights (train-derived):", class_weights)

### 8a. Majority-class baseline — what "doing nothing" scores

Printed before training so the trained model's numbers mean something on sight.

In [ ]:
for col in TASKS:
    for split_name in ["val", "test"]:
        sub = feature_manifest[feature_manifest["split"] == split_name][col].dropna()
        majority_frac = sub.value_counts(normalize=True).max()
        print(f"{col:16s} {split_name:5s}: majority-class baseline accuracy = {majority_frac:.3f}")

### 8b. tf.data pipeline — variable-length features, padded per batch

In [ ]:
def make_generator(split_name):
    sub = feature_manifest[feature_manifest["split"] == split_name].reset_index(drop=True)
    def gen():
        for _, row in sub.iterrows():
            x = np.load(row["npy_path"]).astype(np.float32)
            x = (x - norm_mean) / norm_std
            y, sw = {}, {}
            for col in TASKS:
                val = row[col]
                if pd.isna(val):
                    y[col] = np.float32(0.0)
                    sw[col] = np.float32(0.0)
                else:
                    y[col] = np.float32(val)
                    sw[col] = np.float32(class_weights[col][int(val)])
            yield x, y, sw
    return gen

output_signature = (
    tf.TensorSpec(shape=(None, FEATURE_CONFIG["feature_dim"]), dtype=tf.float32),
    {col: tf.TensorSpec(shape=(), dtype=tf.float32) for col in TASKS},
    {col: tf.TensorSpec(shape=(), dtype=tf.float32) for col in TASKS},
)

def make_dataset(split_name, batch_size=16, shuffle=False):
    ds = tf.data.Dataset.from_generator(make_generator(split_name), output_signature=output_signature)
    if shuffle:
        ds = ds.shuffle(1000, seed=42, reshuffle_each_iteration=True)
    padded_shapes = ([None, FEATURE_CONFIG["feature_dim"]], {c: [] for c in TASKS}, {c: [] for c in TASKS})
    return ds.padded_batch(batch_size, padded_shapes=padded_shapes).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset("train", shuffle=True)
val_ds = make_dataset("val")
test_ds = make_dataset("test")

for x, y, sw in train_ds.take(1):
    print("Batch feature shape:", x.shape)
    print("Batch labels:", {k: v.numpy() for k, v in y.items()})

### 8c. Model — small shared CNN backbone, three sigmoid heads

Deliberately small: Conv1D stack + GlobalAveragePooling1D (handles variable length without needing
mask-aware layers), dropout for the tiny training set. Not a large architecture — per your own
spec, correct data and no leakage matter more than model size at this scale.

In [ ]:
def build_model(feature_dim, tasks):
    inputs = tf.keras.Input(shape=(None, feature_dim), name="mfcc_delta")
    x = tf.keras.layers.Conv1D(32, 5, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = {task: tf.keras.layers.Dense(1, activation="sigmoid", name=task)(x) for task in tasks}
    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = build_model(FEATURE_CONFIG["feature_dim"], TASKS)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={t: "binary_crossentropy" for t in TASKS},
    metrics={t: ["accuracy", tf.keras.metrics.AUC(name="auc")] for t in TASKS},
)
model.summary()

### 8d. Train — checkpointed, early-stopped

`experiment_001_baseline` in the registry sense — a starting point for Phase 11 error analysis to
react to, not a final answer.

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("checkpoints/experiment_001_best_val_loss.keras",
                                        monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    tf.keras.callbacks.CSVLogger("experiment_001_training_history.csv"),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=60, callbacks=callbacks, verbose=2)
model.save("experiment_001_final.keras")
print("Saved checkpoints/experiment_001_best_val_loss.keras, experiment_001_final.keras, "
      "experiment_001_training_history.csv")

### 8e. Test evaluation — against the majority-class baseline, not in isolation

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True, verbose=0)
print(json_module.dumps(test_results, indent=2, default=str))

print("\nModel vs. majority-class baseline on test:")
for col in TASKS:
    acc_key = f"{col}_accuracy"
    baseline = feature_manifest[feature_manifest["split"] == "test"][col].dropna().value_counts(normalize=True).max()
    print(f"  {col:16s} model={test_results.get(acc_key, float('nan')):.3f}  baseline={baseline:.3f}  "
          f"delta={test_results.get(acc_key, float('nan')) - baseline:+.3f}")

## 9. Phase 10 — Full evaluation (precision/recall/F1/confusion matrix)

Real run (experiment_001_baseline): all three tasks beat their majority-class baseline by
+17 to +25 points on test, with AUC well above 0.5 on every task — genuine discriminative learning,
not the model riding `the_tight_noon`'s class imbalance. `separate_tide` is the weakest task, which
lines up with Phase 5's finding that it was the noisiest label in the duplicate-conflict check (14
groups disagreed on it, more than any other field) — a data-quality ceiling, not necessarily a
modeling one. That's a hypothesis for Phase 11 to test, not a conclusion to act on yet.

Threshold is 0.5 here deliberately — that's the default, not a calibrated choice. Phase 13 is where
the threshold gets tuned against the false-positive/false-negative tradeoff; doing it now would
mean re-doing it anyway once the model itself changes.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

y_true = {t: [] for t in TASKS}
y_pred_proba = {t: [] for t in TASKS}
sample_weight_seen = {t: [] for t in TASKS}

for x, y, sw in test_ds:
    preds = model.predict(x, verbose=0)
    for t in TASKS:
        y_true[t].extend(y[t].numpy().tolist())
        y_pred_proba[t].extend(np.asarray(preds[t]).flatten().tolist())
        sample_weight_seen[t].extend(sw[t].numpy().tolist())

phase10_report = {}
for t in TASKS:
    mask = np.array(sample_weight_seen[t]) > 0  # exclude the NaN-labeled clip (sample_weight 0)
    yt = np.array(y_true[t])[mask].astype(int)
    yp = (np.array(y_pred_proba[t])[mask] >= 0.5).astype(int)

    print(f"\n=== {t} (threshold=0.5, n={mask.sum()}) ===")
    cm = confusion_matrix(yt, yp)
    print("Confusion matrix [rows=true 0/1, cols=pred 0/1]:")
    print(cm)
    report_str = classification_report(yt, yp, digits=3, zero_division=0)
    print(report_str)
    bal_acc = balanced_accuracy_score(yt, yp)
    print(f"Balanced accuracy: {bal_acc:.3f}")

    phase10_report[t] = {
        "n": int(mask.sum()),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(yt, yp, digits=3, zero_division=0, output_dict=True),
        "balanced_accuracy": float(bal_acc),
    }

with open("phase10_evaluation.json", "w") as f:
    json_module.dump(phase10_report, f, indent=2, default=str)
print("\nSaved phase10_evaluation.json")

## 10. Phase 11→12 — experiment_002: test the `separate_tide` label-noise hypothesis

Two runs of `experiment_001` now agree: `separate_tide` test accuracy 71.8-73.3%, AUC 0.777-0.806
— consistently the weakest task. Its confusion matrix shows a specific failure mode: recall on the
"incorrect" class is only 59% (55/93 caught, 38 missed) — the model's main error is calling real
mispronunciations "correct." That lines up with Phase 5's finding that `separate_tide` had the most
label disagreements of any field (14 conflicting duplicate-groups).

`experiment_002` changes exactly **one thing** from `experiment_001`: the 14 label-conflicted rows
get zero sample-weight for the `separate_tide` task specifically during training (they still teach
the other two tasks normally). Val/test are untouched — same clips, same labels, same evaluation —
so this is a controlled comparison, not a different experiment. If `separate_tide`'s test metrics
move clearly outside the 71.8-73.3% / AUC 0.78-0.81 band already observed twice, that's real
evidence for the label-noise hypothesis. If not, look elsewhere instead of tuning blind.

In [ ]:
n_conflicted = int(feature_manifest["separate_tide_had_conflict"].sum())
n_conflicted_train = int(feature_manifest.loc[
    (feature_manifest["split"] == "train") & feature_manifest["separate_tide_had_conflict"], :
].shape[0])
print(f"separate_tide label-conflicted rows: {n_conflicted} total, {n_conflicted_train} in train "
      f"(these get zero sample-weight for separate_tide only in experiment_002)")

def make_generator_v2(split_name, exclude_conflicts_task=None):
    sub = feature_manifest[feature_manifest["split"] == split_name].reset_index(drop=True)
    def gen():
        for _, row in sub.iterrows():
            x = np.load(row["npy_path"]).astype(np.float32)
            x = (x - norm_mean) / norm_std
            y, sw = {}, {}
            for col in TASKS:
                val = row[col]
                if pd.isna(val):
                    y[col] = np.float32(0.0)
                    sw[col] = np.float32(0.0)
                elif col == exclude_conflicts_task and bool(row.get(f"{col}_had_conflict", False)):
                    y[col] = np.float32(val)
                    sw[col] = np.float32(0.0)
                else:
                    y[col] = np.float32(val)
                    sw[col] = np.float32(class_weights[col][int(val)])
            yield x, y, sw
    return gen

def make_dataset_v2(split_name, batch_size=16, shuffle=False, exclude_conflicts_task=None):
    ds = tf.data.Dataset.from_generator(make_generator_v2(split_name, exclude_conflicts_task),
                                         output_signature=output_signature)
    if shuffle:
        ds = ds.shuffle(1000, seed=RANDOM_SEED, reshuffle_each_iteration=True)
    padded_shapes = ([None, FEATURE_CONFIG["feature_dim"]], {c: [] for c in TASKS}, {c: [] for c in TASKS})
    return ds.padded_batch(batch_size, padded_shapes=padded_shapes).prefetch(tf.data.AUTOTUNE)

# Val/test stay exactly the same as experiment_001 -- same evaluation, only training changes.
train_ds_v2 = make_dataset_v2("train", shuffle=True, exclude_conflicts_task="separate_tide")

tf.keras.utils.set_random_seed(RANDOM_SEED)  # re-seed so experiment_002's init matches what
# experiment_001 would get if re-run now -- isolates the treatment to the conflict-exclusion only.
model_v2 = build_model(FEATURE_CONFIG["feature_dim"], TASKS)
model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={t: "binary_crossentropy" for t in TASKS},
    metrics={t: ["accuracy", tf.keras.metrics.AUC(name="auc")] for t in TASKS},
)

callbacks_v2 = [
    tf.keras.callbacks.ModelCheckpoint("checkpoints/experiment_002_best_val_loss.keras",
                                        monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    tf.keras.callbacks.CSVLogger("experiment_002_training_history.csv"),
]
history_v2 = model_v2.fit(train_ds_v2, validation_data=val_ds, epochs=60,
                           callbacks=callbacks_v2, verbose=2)
model_v2.save("experiment_002_final.keras")

### 10b. experiment_002 evaluation — same test set, direct comparison

In [ ]:
test_results_v2 = model_v2.evaluate(test_ds, return_dict=True, verbose=0)

print("experiment_001 (baseline) vs experiment_002 (separate_tide conflicts excluded) on test:\n")
for col in TASKS:
    acc_key, auc_key = f"{col}_accuracy", f"{col}_auc"
    print(f"  {col:16s} exp001_acc={test_results.get(acc_key, float('nan')):.3f}  "
          f"exp002_acc={test_results_v2.get(acc_key, float('nan')):.3f}  "
          f"exp001_auc={test_results.get(auc_key, float('nan')):.3f}  "
          f"exp002_auc={test_results_v2.get(auc_key, float('nan')):.3f}")

y_true_v2 = {t: [] for t in TASKS}
y_pred_proba_v2 = {t: [] for t in TASKS}
sw_seen_v2 = {t: [] for t in TASKS}
for x, y, sw in test_ds:
    preds = model_v2.predict(x, verbose=0)
    for t in TASKS:
        y_true_v2[t].extend(y[t].numpy().tolist())
        y_pred_proba_v2[t].extend(np.asarray(preds[t]).flatten().tolist())
        sw_seen_v2[t].extend(sw[t].numpy().tolist())

mask = np.array(sw_seen_v2["separate_tide"]) > 0
yt = np.array(y_true_v2["separate_tide"])[mask].astype(int)
yp = (np.array(y_pred_proba_v2["separate_tide"])[mask] >= 0.5).astype(int)
print("\nexperiment_002 separate_tide confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(yt, yp))
print(classification_report(yt, yp, digits=3, zero_division=0))

## Next step (do not run yet)

Compare experiment_002's `separate_tide` numbers against the 71.8-73.3% / AUC 0.78-0.81 band from
the two experiment_001 runs, and specifically whether the class-0 (real-error) recall improved from
59%. Send back section 10's printed output — that decides whether label-noise is confirmed as the
bottleneck (write it up as a documented data limitation) or ruled out (look at architecture/features
next instead).